# DSB Monitoring Summary CSV Aggregation and QC

This notebook is intended to run **R** code (IRkernel).

Kernel: **R (wilsontew)** (kernelspec: `wilsontew-r`).

Data folder:
- Default: `./PD_Data` (relative to the folder you opened in VS Code)
- Override: set environment variable `WILSONTEW_PD_FOLDER`

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [1]:
# If you just edited .Renviron, you typically need to restart the R kernel for it to take effect.
# As a convenience, this cell can also set the PD folder for the current session.
if (Sys.getenv("WILSONTEW_PD_FOLDER") == "") {
  Sys.setenv(
    WILSONTEW_PD_FOLDER = "C:/Users/dunnmk/University of Michigan Dropbox/MED-WILSONTELAB/wilsontelab box/Common/Projects/Yeast Aim 3/Sequencing/PD Data"
  )
}
message("WILSONTEW_PD_FOLDER=", Sys.getenv("WILSONTEW_PD_FOLDER"))

WILSONTEW_PD_FOLDER=C:/Users/dunnmk/University of Michigan Dropbox/MED-WILSONTELAB/wilsontelab box/Common/Projects/Yeast Aim 3/Sequencing/PD Data



In [3]:
# Local-first path (edit if needed)
pd_folder <- Sys.getenv("WILSONTEW_PD_FOLDER", unset = file.path(getwd(), "PD_Data"))
pd_folder <- normalizePath(pd_folder, winslash = "/", mustWork = FALSE)
if (!dir.exists(pd_folder)) dir.create(pd_folder, recursive = TRUE, showWarnings = FALSE)
message("Using pd_folder: ", pd_folder)

files <- list.files(
  pd_folder,
  pattern = "(_summary|_collapsed_summary)\\.csv$",
  full.names = TRUE
)

if (length(files) == 0) {
  stop(paste0(
    "No *_summary.csv or *_collapsed_summary.csv files found in pd_folder: ", pd_folder,
    "\n\nTo run locally, either copy your PD summary CSVs into that folder,",
    " or set WILSONTEW_PD_FOLDER to the directory that contains them."
  ))
}

# Read + annotate without relying on %>% being available
dat_raw <- purrr::map_dfr(
  files,
  ~ dplyr::mutate(
    readr::read_csv(.x, show_col_types = FALSE),
    source_path = .x
  )
 )

dplyr::glimpse(dat_raw)

Using pd_folder: C:/Users/dunnmk/University of Michigan Dropbox/MED-WILSONTELAB/wilsontelab box/Common/Projects/Yeast Aim 3/Sequencing/PD Data



Rows: 659
Columns: 12
$ batch          <dbl> 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6…
$ DSB2_loci      <chr> "chr4", "chr4", "chr4", "chr4", "chr4", "chr4", "chr4",…
$ time_point     <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0…
$ replicate      <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1…
$ alignment_name <chr> "CIS_DSB1_FULL_CHRIII_L02", "CIS_DSB1_FULL_CHRIII_L03",…
$ cis_trans      <chr> "CIS", "CIS", "CIS", "CIS", "CIS", "CIS", "CIS", "CIS",…
$ DSB            <chr> "DSB1", "DSB1", "DSB1", "DSB1", "DSB1", "DSB1", "DSB1",…
$ combo          <chr> "A_to_B", "A_to_B", "A_to_B", "A_to_B", "A_to_B", "A_to…
$ `repeat`       <chr> "INTACT", "INTACT", "INTACT", "INTACT", "INTACT", "INTA…
$ allele         <chr> "CHRIII_L02", "CHRIII_L03", "CHRIII_L04", "CHRIV_L01", …
$ count          <dbl> 51283, 38174, 228463, 49541, 32857, 54088, 39551, 44777…
$ source_path    <chr> "C:/Users/dunnmk/University of Michigan Dropbox/MED-WIL…
